# 第 11 章 サポートベクターマシンとカーネル法

できるだけ広い余白（マージン）を空ける境界線を選びます。カーネルで XOR も解きます。

対応する記事: [第 11 章 サポートベクターマシンとカーネル法（Kotlin 版）](https://github.com/k2works/grokking-machine-learning-excersice/blob/main/docs/article/grokking-machine-learning/kotlin/ch11.md)

実装本体: `apps/grokking-ml-kotlin/src/`

## セットアップ

実装本体をビルドした JAR を読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

先に JAR を作っておいてください。

```bash
cd apps/grokking-ml-kotlin
./gradlew jar
```

IntelliJ IDEA の Kotlin Notebook プラグイン、または [Kotlin Jupyter カーネル](https://github.com/Kotlin/kotlin-jupyter) で開きます。

```bash
pip install kotlin-jupyter-kernel
jupyter lab notebooks/
```

In [1]:
@file:DependsOn("../build/libs/grokking-ml-kotlin-0.1.0.jar")

import ch05.*
import ch11.*

## パーセプトロンのマージンは 0

第 5 章のパーセプトロンは、分離できた時点で更新を止めます。**その境界線がぎりぎりでも構いません。**

実際に測ると、境界線の上にちょうど乗っている点があります。正解率は 1.0 なのに、その点が少しでも動けば誤分類になります。

In [2]:
val points = listOf(listOf(1.0, 0.0), listOf(0.0, 2.0), listOf(1.0, 1.0), listOf(1.0, 2.0),
                    listOf(1.0, 3.0), listOf(2.0, 2.0), listOf(2.0, 3.0), listOf(3.0, 2.0))
val labels = listOf(-1, -1, -1, -1, 1, 1, 1, 1)
val perceptronLabels = listOf(0, 0, 0, 0, 1, 1, 1, 1)

val (perceptron, _) = ch05.perceptronAlgorithm(points, perceptronLabels)
val asSvm = SupportVectorMachine(perceptron.weights, perceptron.bias)

println("パーセプトロン 正解率 %.2f  マージン %.4f".format(accuracy(asSvm, points, labels), asSvm.margin(points)))

パーセプトロン 正解率 1.00  マージン 0.0000


## SVM は余白を稼ぐ

**正解率は同じ 1.0 でも、マージンがまったく違います。** ヒンジ損失が「正解しているのにマージンの内側にいる点」も押し返すためです。

In [3]:
val (svm, errors) = trainSvm(points, labels, epochs = 20000, regularization = 0.01)

println("重み   " + svm.weights.map { "%.4f".format(it) })
println("バイアス %.4f".format(svm.bias))
println("正解率  %.2f".format(accuracy(svm, points, labels)))
println("マージン %.4f".format(svm.margin(points)))

重み   [1.7261, 1.6864]
バイアス -5.7800
正解率  1.00
マージン 0.5644


## ヒンジ損失は「正解しているのに損失が残る」

第 5 章のパーセプトロン誤差は正解した点を無視しました。**ヒンジ損失はマージンの外に出るまで押し続けます。**

In [4]:
val sample = SupportVectorMachine(listOf(1.0, 1.0), -3.0)

println("%-12s %8s %6s %8s".format("点", "スコア", "ラベル", "損失"))
listOf(listOf(3.0, 2.0), listOf(1.5, 2.0), listOf(2.0, 1.0), listOf(1.0, 1.0)).forEach { point ->
    println("%-12s %8.1f %6d %8.2f".format(point, sample.score(point), 1, hingeLoss(sample, point, 1)))
}

点                 スコア    ラベル       損失
[3.0, 2.0]        2.0      1     0.00


[1.5, 2.0]        0.5      1     0.50
[2.0, 1.0]        0.0      1     1.00


[1.0, 1.0]       -1.0      1     2.00


## カーネルで XOR を解く

**アルゴリズムは一切変えず、カーネル関数を差し替えるだけ** で XOR が解けます。線形カーネル（ただの内積）では解けません。

In [5]:
val xorPoints = listOf(listOf(0.0, 0.0), listOf(0.0, 1.0), listOf(1.0, 0.0), listOf(1.0, 1.0))
val xorLabels = listOf(-1, 1, 1, -1)

listOf("線形" to linearKernel, "多項式 2 次" to polynomialKernel(2), "RBF" to rbfKernel(1.0))
    .forEach { (name, kernel) ->
        val m = trainKernelClassifier(xorPoints, xorLabels, kernel = kernel)
        println("%-12s 正解率 %.2f".format(name, kernelAccuracy(m, xorPoints, xorLabels)))
    }

線形           正解率 0.75
多項式 2 次      正解率 1.00
RBF          正解率 1.00


## 試してみる: 正則化とマージン

**直感に反しますが、正則化を強くするとマージンは狭くなります。** 重みを潰しすぎると、境界線からの距離そのものが縮むためです。

In [6]:
listOf(0.005, 0.01, 0.05, 0.1, 0.5).forEach { strength ->
    val (m, _) = trainSvm(points, labels, epochs = 20000, regularization = strength)
    println("λ = %-6s マージン %.4f  正解率 %.2f".format(strength, m.margin(points),
            accuracy(m, points, labels)))
}

λ = 0.005  マージン 0.6510  正解率 1.00
λ = 0.01   マージン 0.5644  正解率 1.00


λ = 0.05   マージン 0.0214  正解率 1.00


λ = 0.1    マージン 0.2057  正解率 1.00
λ = 0.5    マージン 0.2429  正解率 1.00
